<a href="https://colab.research.google.com/github/Huy1902/HAC/blob/main/Movie_Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np 
import pandas as pd 
import os

In [2]:
path_to_data = '/kaggle/input/music-interaction/interaction.csv'
path_to_item = 'dataset/tracks_features.csv'
path_to_output = 'dataset/'

## Extract train and test dataset

In [7]:
data = pd.read_csv(path_to_data, sep=',')
data

,user_id,item_id,exists,timestamp
0,mpd_pid_0,6I9VzXrHxO9rA9A5euc8Ak,1,1493423880
1,mpd_pid_0,1AWQoqb9bSvzTjaLralEkT,1,1493423640
2,mpd_pid_0,7H6ev70Weq6DdpZyyTmUXk,1,1493423040
3,mpd_pid_0,2PpruBYCo4H7WOBJ7Q2EwM,1,1493416800
4,mpd_pid_0,4pmc2AxSEq6g7hPVlJCPyP,1,1493409480
...,...,...,...,...
11808549,mpd_pid_999996,2NrhLJdQJaga0YAI8Ipl16,0,1475599380
11808550,mpd_pid_999997,0pziAK2FOGW0h5m5gjMYAB,0,1498168380
11808551,mpd_pid_999997,46guedJC6IiQ8jsF3WkVLx,0,1498168380
11808552,mpd_pid_999997,4nKLRpFaGWQ6dwI5gaSys6,0,1498168380


In [8]:
data['user_click'] = data['exists']
data = data.sort_values(by=['user_id', 'timestamp']).reset_index(drop=True)
result = []
test_df = []
train_df = []
for user_id, user_data in data.groupby('user_id'):
    user_data = user_data.reset_index(drop=True)
    click_history = []
    seq_id = 0
    sep = int(0.9 * len(user_data)) // 10
    # Split into segments of 10 items
    for i in range(0, len(user_data), 10):
        if(i + 10 > len(user_data)):
            break
        segment = user_data[i:i+10]
        slate_of_items = segment['item_id'].tolist()
        user_clicks = segment['user_click'].tolist()

        row = {
          'user_id': user_id,
          'slate_of_items': slate_of_items,
          'user_mid': user_clicks,
          'user_mid_history': click_history[:],
          'sequence_id': seq_id
        }
        result.append(row)
        if seq_id < sep:
            train_df.append(row)
        else:
            test_df.append(row)
        # Update click history
        click_history.extend(segment[segment['user_click'] == 1]['item_id'].tolist())
        seq_id += 1

train_df = pd.DataFrame(train_df)
test_df = pd.DataFrame(test_df)
result_df = pd.DataFrame(result)
os.makedirs(path_to_output, exist_ok=True)
train_df.to_csv(os.path.join(path_to_output, 'train.csv') , sep='@', index=False)
test_df.to_csv(os.path.join(path_to_output, 'test.csv') , sep='@', index=False)

result_df.to_csv(os.path.join(path_to_output, 'all.csv') , sep='@', index=False)

## Extract item_info

In [9]:
train_df.info()
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 626140 entries, 0 to 626139
Data columns (total 5 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   user_id           626140 non-null  object
 1   slate_of_items    626140 non-null  object
 2   user_mid          626140 non-null  object
 3   user_mid_history  626140 non-null  object
 4   sequence_id       626140 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 23.9+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200574 entries, 0 to 200573
Data columns (total 5 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   user_id           200574 non-null  object
 1   slate_of_items    200574 non-null  object
 2   user_mid          200574 non-null  object
 3   user_mid_history  200574 non-null  object
 4   sequence_id       200574 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 7.7+ MB


In [4]:
df = pd.read_csv(path_to_item)

In [14]:
df[df['id'] == '62sebgxLnupowQ9KULc11J']

,id,name,album,album_id,artists,artist_ids,track_number,disc_number,explicit,danceability,...,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,time_signature,year,release_date
684725,62sebgxLnupowQ9KULc11J,"She's Got a Way - Live at the Paradise, Boston...",Songs In the Attic,2Vf4bohoWVk1YlPR2uNOFd,['Billy Joel'],['6zFYqv1mOsgBRQbae3JJ9e'],5,1,False,0.414,...,0.036,0.969,0.00069,0.853,0.423,70.524,180600,4.0,1981,1981-09-14


In [5]:

import pandas as pd
import ast

# --- inputs ---
# If you already have df = pd.read_csv(path_to_item) above, keep it; otherwise uncomment next line:
# df = pd.read_csv("path/to/your_tracks_dump.csv")

track_ids = [
    '78WVLOP9pN0G3gRLFy1rAa', '78RIER8V6EhrqVPOBi2GYa', '16x9viSmRS3PII71Pdeowc',
    '7BvuV4c1BGhaRapcXvRu5z', '1SRkKyJ2JjMZgyDWC30zKv', '4alHo6RGd0D3OUbTPExTHN',
    '1BhMUHjiQlwGlT5ZaMHHSf', '2PpruBYCo4H7WOBJ7Q2EwM', '5WwqdeavrQrbeAMDxGawse',
    '0cO7JEo8deKuQMWpDyjenY', '0weAUscowxeqDtpCgtbpgp', '1C4yOUEjCUyPkxRDwwFksG',
    '28clONjZmul6FjfO6tZQDE', '1G391cbiT3v3Cywg8T7DM1', '27AHAtAirQapVldIm4c9ZX',
    '5AMvBCtX2rspUdoeJ9IsPN', '5jSz894ljfWE0IcHBSM39i', '5gys5nzVQIYhgHIfiOJYva',
    '3bE5slaVEfaDreqARl6k4M', '2J6QnTjHIWwXErNWyF0RUC', '6jBCehpNMkwFVF3dz4nLIW',
    '6qUEOWqOzu1rLPUPQ1ECpx', '71SvEDmsOwIWw1IozsZoMA', '2aoo2jlRnM3A0NyLQqMN2f',
    '08mG3Y1vljYA6bvDt4Wqkj'
]

# --- helper to normalize artists column into a single string like "A/B/Ft. C" (here we just join with "/") ---
def normalize_artists(x):
    # Already a list of strings?
    if isinstance(x, list):
        return "/".join(str(s).strip() for s in x)

    # String that might be "['A', 'B']" or a JSON-ish list or list of dicts
    if isinstance(x, str):
        s = x.strip()
        try:
            val = ast.literal_eval(s)
            if isinstance(val, list):
                # list of strings OR list of dicts with "name"
                names = []
                for item in val:
                    if isinstance(item, dict) and "name" in item:
                        names.append(str(item["name"]).strip())
                    else:
                        names.append(str(item).strip())
                return "/".join(names)
        except Exception:
            # not parseable, keep as-is
            return s
        return s

    # Fallback
    return str(x)

# --- filter & order to match given ID sequence ---
pos = {tid: i for i, tid in enumerate(track_ids)}
sub = df[df["id"].isin(track_ids)].copy()
sub["__pos"] = sub["id"].map(pos)

# build final two columns
sub["track_name"]  = sub["name"]
sub["artist_name"] = sub["artists"].apply(normalize_artists)


# ... keep everything from the previous snippet above ...

# build final three columns (keep input order)
out = sub.sort_values("__pos")[["id", "track_name", "artist_name", "album"]]

# write with header
out.to_csv("tracks_with_id.csv", index=False, encoding="utf-8")
print("Saved: tracks_with_id.csv")

# ---- If you need the special first row with 'Standard' and NO header (like your screenshot) ----
# out2 = pd.concat(
#     [pd.DataFrame([{"id":"Standard","track_name":"Standard","artist_name":"Standard"}]), out],
#     ignore_index=True
# )
# out2.to_csv("tracks_with_id.csv", index=False, header=False, encoding="utf-8")



Saved: tracks_with_id.csv


In [10]:
os.makedirs(path_to_output, exist_ok=True)

FEATURE_COLS = [
    "danceability","energy","key","loudness","mode","speechiness",
    "acousticness","instrumentalness","liveness","valence",
    "tempo","duration_ms","time_signature","year"
]

df = pd.read_csv(path_to_item)
missing = [c for c in ["id"] + FEATURE_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in tracks CSV: {missing}")

# optional min–max scale
feats = df[FEATURE_COLS].astype(float).copy()
for c in FEATURE_COLS:
    col = feats[c]
    cmin, cmax = float(col.min()), float(col.max())
    feats[c] = 0.0 if cmax == cmin else (col - cmin) / (cmax - cmin)
item_info = feats.astype(np.float32).values  # (N, D)

# save item_ids as fixed-width unicode (no pickle needed)
ids = df["id"].astype(str)
max_len = int(ids.str.len().max())
item_ids = ids.to_numpy(dtype=f"<U{max_len}")  # (N,)

np.save(os.path.join(path_to_output, "item_info.npy"), item_info)
np.save(os.path.join(path_to_output, "item_ids.npy"),  item_ids)

print("Saved:",
      os.path.join(path_to_output, "item_info.npy"), item_info.shape,
      "|", os.path.join(path_to_output, "item_ids.npy"), item_ids.shape)
print("Feature order:", FEATURE_COLS)



Saved: /kaggle/working/item_info.npy (1204025, 14) | /kaggle/working/item_ids.npy (1204025,)
Feature order: ['danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'time_signature', 'year']


In [9]:
item_meta = np.load(os.path.join(path_to_output, "item_info.npy"))
item_meta.shape

(1204025, 14)

In [12]:
interactions_csv = os.path.join(path_to_data)
filtered_interactions_csv = os.path.join(path_to_output, "interactions_filtered.csv")

item_ids = np.load(os.path.join(path_to_output, "item_ids.npy"))
allowed = set(item_ids.tolist())

inter = pd.read_csv(interactions_csv)
if "item_id" not in inter.columns:
    raise ValueError("interactions CSV must have an 'item_id' column")

inter["item_id"] = inter["item_id"].astype(str)
before = len(inter)
inter_f = inter[inter["item_id"].isin(allowed)].copy()
after = len(inter_f)

inter_f.to_csv(filtered_interactions_csv, index=False)
print(f"Kept {after}/{before} interactions ({after/before:.2%}).")
print("Saved ->", filtered_interactions_csv)


Kept 11808554/11808554 interactions (100.00%).
Saved -> /kaggle/working/interactions_filtered.csv


## Extract user info (just id)

In [13]:
data.head()

,user_id,item_id,exists,timestamp,user_click
0,mpd_pid_0,19Js5ypV6JKn4DMExHQbGc,1,1493409120,1
1,mpd_pid_0,4pmc2AxSEq6g7hPVlJCPyP,1,1493409480,1
2,mpd_pid_0,2PpruBYCo4H7WOBJ7Q2EwM,1,1493416800,1
3,mpd_pid_0,25p5r2BN3lfA9r4a24qSVx,0,1493419920,0
4,mpd_pid_0,5avWeHRXgA7gJPUOLu2WxA,0,1493419920,0


In [18]:
set_of_user = list(set(data['user_id'].to_list()))

In [19]:
len(set_of_user)

887060

In [20]:
user_info = np.array(set_of_user).astype(str)
np.save(os.path.join(path_to_output, "user_info.npy"), user_info)
user_info.shape

(887060,)

In [6]:
user_meta = np.load(os.path.join(path_to_output, "user_info.npy"), allow_pickle=True)
user_meta

FileNotFoundError: [Errno 2] No such file or directory: '/dataset/user_info.npy'

In [10]:
item_meta

array([[0.47      , 0.978     , 0.6363636 , ..., 0.03450988, 0.8       ,
        0.98960394],
       [0.599     , 0.957     , 1.        , ..., 0.03386088, 0.8       ,
        0.98960394],
       [0.315     , 0.97      , 0.6363636 , ..., 0.04915653, 0.8       ,
        0.98960394],
       ...,
       [0.785     , 0.796     , 0.8181818 , ..., 0.06342068, 0.8       ,
        0.9970297 ],
       [0.665     , 0.856     , 0.54545456, ..., 0.05337462, 0.8       ,
        0.9970297 ],
       [0.736     , 0.708     , 0.18181819, ..., 0.0501613 , 0.8       ,
        0.9970297 ]], shape=(1204025, 14), dtype=float32)